# MEDIA Analytics — Pipeline Fabric (SIH/SUS)
**Versão 1.0 — Bronze → Silver → Gold → Power BI**

Este notebook roda no Microsoft Fabric com um **Lakehouse padrão anexado**. Trabalha inicialmente com arquivos CSV ou Parquet reais de internações SIH/SUS. Não fabrica dados nem assume que um repositório GitHub hospede os datasets. Configure `SOURCE_PATH` para um arquivo em `Files/media/inbox/` antes de executar a ingestão.

**Atenção:** a base SIH/SUS cobre internações financiadas pelo SUS; não representa todas as internações brasileiras. A classificação cardiovascular é feita pelo diagnóstico principal CID-10 capítulo IX (I00–I99), conforme disponível no arquivo.

## 01 — Configuração e diagnóstico

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re
from pyspark.sql import functions as F, types as T

ROOT = Path('/lakehouse/default/Files/media')
if not Path('/lakehouse/default/Files').exists():
    raise RuntimeError('Anexe um Lakehouse como PADRÃO ao notebook Fabric e reinicie a sessão.')
for sub in ['inbox','bronze/sih','logs']:
    (ROOT/sub).mkdir(parents=True, exist_ok=True)
print('Spark:', spark.version)
print('Lakehouse:', ROOT)
print('Execução UTC:', datetime.now(timezone.utc).isoformat())

## 02 — Configurar o arquivo de origem
Faça upload de um CSV ou Parquet oficial do SIH/SUS (ou de um derivado com proveniência documentada) em **Lakehouse → Files → media → inbox**. Inspecione o nome e altere `SOURCE_PATH`. CSV do SIH costuma ter codificação Latin-1 e delimitador `;`, mas confirme no arquivo.

In [ ]:
for p in sorted((ROOT/'inbox').iterdir()):
    print(p.name, p.stat().st_size, 'bytes')

SOURCE_PATH = ''  # EXEMPLO: '/lakehouse/default/Files/media/inbox/RD_MS_2024.csv'
SOURCE_FORMAT = 'csv'  # 'csv' ou 'parquet'
CSV_SEPARATOR = ';'
CSV_ENCODING = 'ISO-8859-1'
SOURCE_ID = 'SIH_SUS'  # Identificação/proveniência

if not SOURCE_PATH:
    print('Configure SOURCE_PATH para um arquivo real antes de executar as próximas células.')

## 03 — Bronze: leitura, rastreabilidade e preservação

In [ ]:
if not SOURCE_PATH or not Path(SOURCE_PATH).is_file():
    raise FileNotFoundError(f'Arquivo de origem não encontrado: {SOURCE_PATH!r}')
if SOURCE_FORMAT == 'csv':
    raw = (spark.read.option('header',True).option('sep',CSV_SEPARATOR)
           .option('encoding',CSV_ENCODING).option('inferSchema',False).csv(SOURCE_PATH))
elif SOURCE_FORMAT == 'parquet':
    raw = spark.read.parquet(SOURCE_PATH)
else:
    raise ValueError('SOURCE_FORMAT deve ser csv ou parquet')
raw = raw.toDF(*[c.strip().upper() for c in raw.columns])
if len(set(raw.columns)) != len(raw.columns):
    raise ValueError('Colunas duplicadas após normalização dos cabeçalhos')
raw = (raw.withColumn('_source_file', F.lit(Path(SOURCE_PATH).name))
       .withColumn('_source_id', F.lit(SOURCE_ID))
       .withColumn('_ingested_at_utc', F.current_timestamp()))
print('Colunas:', raw.columns)
print('Registros:', raw.count())
display(raw.limit(10))

In [ ]:
# Tabela Bronze versionada por carga (append). Evita sobrescrever o histórico.
raw.write.format('delta').mode('append').option('mergeSchema','true').saveAsTable('media_bronze_sih')
print('Tabela media_bronze_sih atualizada')

## 04 — Silver: padronização SIH/SUS
O código requer `DIAG_PRINC` e `MUNIC_RES` (campos comuns nos arquivos reduzidos RD). O período de competência deve vir de `ANO_CMPT` e `MES_CMPT`, ou de `DT_INTER` (data de internação). Não supomos que um arquivo genérico seja SIH sem conferir seu esquema.

In [ ]:
required = {'DIAG_PRINC','MUNIC_RES'}
missing = required-set(raw.columns)
if missing:
    raise ValueError(f'Campos SIH obrigatórios ausentes: {sorted(missing)}. Confira o layout da fonte.')
if not ({'ANO_CMPT','MES_CMPT'} <= set(raw.columns) or 'DT_INTER' in raw.columns):
    raise ValueError('É necessário ANO_CMPT + MES_CMPT ou DT_INTER para definir o período.')

clean = raw
for c in ['DIAG_PRINC','MUNIC_RES','ANO_CMPT','MES_CMPT','VAL_TOT','N_AIH','DT_INTER']:
    if c in clean.columns:
        clean = clean.withColumn(c, F.trim(F.col(c).cast('string')))

if {'ANO_CMPT','MES_CMPT'} <= set(clean.columns):
    clean = (clean.withColumn('ano', F.col('ANO_CMPT').cast('int'))
                  .withColumn('mes', F.col('MES_CMPT').cast('int')))
else:
    clean = clean.withColumn('data_internacao', F.to_date('DT_INTER','yyyyMMdd'))
    clean = (clean.withColumn('ano',F.year('data_internacao'))
                  .withColumn('mes',F.month('data_internacao')))

clean = (clean.withColumn('cid10',F.upper(F.regexp_replace('DIAG_PRINC',r'[^A-Za-z0-9]','')))
              .withColumn('municipio_residencia_codigo',F.col('MUNIC_RES'))
              .withColumn('cardiovascular',F.col('cid10').rlike(r'^I[0-9]{2}')))
if 'VAL_TOT' in clean.columns:
    clean = clean.withColumn('valor_total',F.regexp_replace('VAL_TOT',',','.').cast('decimal(18,2)'))
else:
    clean = clean.withColumn('valor_total',F.lit(None).cast('decimal(18,2)'))

clean = clean.filter(F.col('ano').between(2000,2100) & F.col('mes').between(1,12))
clean = clean.filter(F.col('municipio_residencia_codigo').isNotNull() & (F.length('municipio_residencia_codigo')>0))
clean = clean.filter(F.col('cid10').rlike(r'^[A-Z][0-9]{2}'))

# Recorte do estudo: diagnóstico principal cardiovascular (CID-10 I00-I99)
silver = clean.filter(F.col('cardiovascular'))
print('Registros cardiovasculares:',silver.count())
display(silver.select('ano','mes','municipio_residencia_codigo','cid10','valor_total').limit(10))

In [ ]:
# Primeira versão: reconstrução da Silver a partir da Bronze completa,
# evitando duplicação em reexecuções do mesmo arquivo.
# Em produção, aplicar deduplicação por identificador de AIH e competência,
# depois de validar o layout e a semântica do identificador.
bronze_count = spark.table('media_bronze_sih').count()
print('Total bruto armazenado (inclui eventuais reprocessamentos):',bronze_count)
# Aqui persistimos apenas o lote validado. Para múltiplos lotes, criar merge por chave natural validada.
silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('media_silver_sih_cardio')
print('Silver salva: media_silver_sih_cardio')

## 05 — Gold: dimensões e fato agregado
A granularidade da fato é **ano × mês × município de residência × CID-10**. O código municipal SIH não é automaticamente uma chave IBGE de sete dígitos: a correspondência precisa ser validada antes do mapa municipal.

In [ ]:
s = spark.table('media_silver_sih_cardio')

dim_tempo = (s.select('ano','mes').distinct()
    .withColumn('tempo_key',F.col('ano')*100+F.col('mes')))
dim_localidade = (s.select('municipio_residencia_codigo').distinct()
    .withColumnRenamed('municipio_residencia_codigo','localidade_key'))
dim_cid = (s.select('cid10').distinct().withColumnRenamed('cid10','cid_key'))

fato = (s.groupBy('ano','mes','municipio_residencia_codigo','cid10')
    .agg(F.count(F.lit(1)).alias('qtd_registros_aih'),
         F.sum('valor_total').alias('valor_total_sus'))
    .withColumn('tempo_key',F.col('ano')*100+F.col('mes'))
    .withColumnRenamed('municipio_residencia_codigo','localidade_key')
    .withColumnRenamed('cid10','cid_key'))

for name,df in [('media_dim_tempo',dim_tempo),('media_dim_localidade',dim_localidade),
                ('media_dim_cid',dim_cid),('media_fato_internacoes_cardio',fato)]:
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(name)
    print(name, df.count())

display(fato.orderBy(F.desc('qtd_registros_aih')).limit(20))

## 06 — Testes de qualidade e reconciliação
A quantidade representa **registros de AIH na fonte carregada**, não pacientes únicos. Para afirmar quantidade de internações, precisamos validar regras de continuidade, duplicidade e o significado de cada registro no SIH.

In [ ]:
total_silver = s.count()
total_gold = fato.agg(F.sum('qtd_registros_aih').alias('n')).first()['n'] or 0
assert total_silver == total_gold, f'Divergência Silver={total_silver}, Gold={total_gold}'
assert fato.filter(F.col('qtd_registros_aih')<0).count()==0
assert fato.filter(F.col('tempo_key').isNull() | F.col('localidade_key').isNull() | F.col('cid_key').isNull()).count()==0
print('Reconciliação OK:',total_silver,'registros')

log = spark.createDataFrame([(
    datetime.now(timezone.utc).isoformat(),SOURCE_ID,Path(SOURCE_PATH).name,
    int(total_silver),int(total_gold),'OK')],
    ['executado_em_utc','fonte','arquivo','registros_silver','registros_gold','status'])
log.write.format('delta').mode('append').saveAsTable('media_log_pipeline')
display(log)

## 07 — Power BI: modelo semântico

No workspace Fabric, abra o Lakehouse e crie/edite um **modelo semântico** com:

- `media_fato_internacoes_cardio[tempo_key]` → `media_dim_tempo[tempo_key]` (N:1)
- `media_fato_internacoes_cardio[localidade_key]` → `media_dim_localidade[localidade_key]` (N:1)
- `media_fato_internacoes_cardio[cid_key]` → `media_dim_cid[cid_key]` (N:1)

Medidas DAX sugeridas:

```dax
Registros AIH = SUM(media_fato_internacoes_cardio[qtd_registros_aih])
Valor SUS = SUM(media_fato_internacoes_cardio[valor_total_sus])
```

**Próximos incrementos:** mapear códigos SIH → IBGE; adicionar população por município e ano (fonte IBGE, com compatibilidade temporal) e então calcular taxas. Adicionar SIM com definição epidemiológica e cobertura documentadas. Não calcular taxa por 100 mil sem denominador validado.

**Reexecução:** Bronze usa `append` e pode duplicar se você rodar a mesma carga mais de uma vez; Silver e Gold sobrescrevem a versão atual. Para produção, adicionar manifesto e deduplicação de lotes por checksum e chave de negócio validada.